# 02 — Classification Automatique des Tickets (NLP)
## Helpdesk IA — GEISER | PFE 2026

### Objectif
Entraîner un modèle de Machine Learning capable de :
- **Classifier automatiquement** la catégorie d'un ticket
- **Prédire la priorité** d'un ticket

### Pipeline NLP

### Modèles comparés
- Logistic Regression
- LinearSVC
- Random Forest
- Naive Bayes
- XGBoost

### Dataset
- **Source** : dataset_labeled.csv (produit par notebook 01)
- **Taille** : ~1915 tickets GEISER
- **Features** : text (OBJET + DESCRIPTION combinés)
- **Cibles** :
  - `categorie` (10 classes)
  - `sous_categorie` (39 classes)
  - `priorite` (4 classes)

---
## 1. Imports

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# NLP & ML
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

# Modèles
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier

# MLflow — carnet de bord des entraînements (nouveau)
import mlflow
mlflow.set_experiment("helpdesk-ia-classification")

print("✅ Imports OK")
print(f"Scikit-learn : {__import__('sklearn').__version__}")
print(f"MLflow       : {mlflow.__version__} — experiment 'helpdesk-ia-classification' prêt")

---
## 2. Chargement du Dataset

In [ ]:
# ============================================================
# CHARGEMENT DATASET LABELED
# ============================================================
df = pd.read_csv('../../data/dataset_labeled.csv', encoding='utf-8')

print(f"✅ Dataset chargé : {df.shape[0]} tickets, {df.shape[1]} colonnes")
print(f"\nColonnes : {df.columns.tolist()}")
print(f"\n=== DISTRIBUTION CATÉGORIES ({df['categorie'].nunique()} classes) ===")
print(df['categorie'].value_counts())
print(f"\n=== DISTRIBUTION SOUS-CATÉGORIES ({df['sous_categorie'].nunique()} classes) ===")
print(df['sous_categorie'].value_counts().head(10))
print(f"\n=== DISTRIBUTION PRIORITÉS ===")
print(df['priorite'].value_counts())

---
## 3. Préparation des Données

### Fonction de nettoyage
On réapplique le nettoyage pour garantir la qualité du texte
avant vectorisation TF-IDF.

on transforme le texte brut en texte 100% propre pour que TF-IDF puisse compter les mots correctement. Sans ça, "Réseau", "réseau", "réseau!" seraient comptés comme 3 mots différents.

### TF-IDF — Pourquoi ce choix ?
**TF-IDF** (Term Frequency - Inverse Document Frequency) :
- Transforme le texte en vecteurs numériques
- Donne plus de poids aux mots **importants et rares**
- Ignore les mots trop fréquents (stop words)
- Léger et efficace pour des corpus de taille moyenne

### Paramètres choisis
| Paramètre | Valeur | Raison |
|---|---|---|
| max_features | 5000 | Limiter la dimensionnalité |
| ngram_range | (1,2) | Capturer les bigrammes |
| min_df | 2 | Ignorer les mots rares |
| sublinear_tf | True | Normaliser les fréquences |

In [ ]:
# ============================================================
# NETTOYAGE TEXTE
# ============================================================
def clean_text(text):
    """Nettoyer et normaliser le texte pour TF-IDF"""
    if pd.isna(text) or str(text).strip() == '':
        return ''
    text = str(text).lower()
    text = re.sub(r'<[^>]+>', ' ', text)       # HTML tags
    text = re.sub(r'&\w+;', ' ', text)          # HTML entities
    text = re.sub(r'http\S+', ' ', text)         # URLs
    text = re.sub(r'\b\d+\b', ' ', text)         # chiffres isolés
    text = re.sub(r'[^a-zà-ÿ\s]', ' ', text)   # caractères spéciaux
    text = re.sub(r'\s+', ' ', text).strip()     # espaces multiples
    return text

# Appliquer nettoyage
df['text_clean'] = df['text'].apply(clean_text)

# Supprimer lignes vides
df = df[df['text_clean'].str.len() > 5].reset_index(drop=True)

print(f"✅ Texte nettoyé : {df.shape[0]} tickets")
print(f"\nExemple avant : {df['text'].iloc[0][:100]}")
print(f"Exemple après : {df['text_clean'].iloc[0][:100]}")

---
## 4. Classification des Catégories

### Étapes
1. Encoder les labels (texte → entier)
2. Split train/test (80% / 20%)
3. Entraîner 5 modèles avec pipeline TF-IDF
4. Comparer les performances
5. Sélectionner le meilleur modèle

In [ ]:
# ============================================================
# PRÉPARATION FEATURES ET LABELS — CATÉGORIE
# ============================================================
X = df['text_clean']

# Encoder labels catégorie
le_cat = LabelEncoder()
y_cat = le_cat.fit_transform(df['categorie'])

print(f"Classes catégorie ({len(le_cat.classes_)}) :")
for i, cls in enumerate(le_cat.classes_):
    count = (df['categorie'] == cls).sum()
    print(f"  {i:2d} — {cls:<20} : {count} tickets")

# Split 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y_cat,
    test_size=0.2,
    random_state=42,
    stratify=y_cat
)

print(f"\n✅ Split effectué :")
print(f"   Train : {len(X_train)} tickets (80%)")
print(f"   Test  : {len(X_test)} tickets (20%)")

In [ ]:
# ============================================================
# ENTRAÎNEMENT 5 MODÈLES — CATÉGORIE
# ============================================================

# Définition des modèles
models = {
    'Logistic Regression': LogisticRegression(
        max_iter=1000, random_state=42, C=1.0
    ),
    'LinearSVC': LinearSVC(
        random_state=42, max_iter=2000
    ),
    'Random Forest': RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    ),
    'Naive Bayes': MultinomialNB(alpha=0.1),
    'XGBoost': XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        random_state=42, eval_metric='mlogloss', n_jobs=-1
    ),
}

# TF-IDF config
tfidf_params = dict(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True
)

results_cat = {}

print("=== ENTRAÎNEMENT — CATÉGORIE ===\n")

for name, model in models.items():
    with mlflow.start_run(run_name=f"categorie-{name}"):
        # Pipeline TF-IDF + Modèle
        pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(**tfidf_params)),
            ('clf', model)
        ])

        # Entraînement
        pipeline.fit(X_train, y_train)

        # Prédiction
        y_pred = pipeline.predict(X_test)

        # Métriques
        acc = accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='weighted')

        results_cat[name] = {
            'pipeline': pipeline,
            'accuracy': acc,
            'f1_score': f1,
            'y_pred': y_pred
        }

        # MLflow — on note ce run dans le carnet de bord
        mlflow.set_tag("cible", "categorie")
        mlflow.log_param("modele", name)
        mlflow.log_params(tfidf_params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)

        print(f"✅ {name:<25} Accuracy: {acc:.4f}  F1: {f1:.4f}")

In [ ]:
# ============================================================
# COMPARAISON VISUELLE — CATÉGORIE
# ============================================================
names = list(results_cat.keys())
accuracies = [results_cat[n]['accuracy'] for n in names]
f1_scores  = [results_cat[n]['f1_score'] for n in names]

x = np.arange(len(names))
width = 0.35

fig, ax = plt.subplots(figsize=(12, 5))
bars1 = ax.bar(x - width/2, accuracies, width,
               label='Accuracy', color='#534AB7', alpha=0.85)
bars2 = ax.bar(x + width/2, f1_scores, width,
               label='F1-Score', color='#1D9E75', alpha=0.85)

ax.set_xlabel('Modèle')
ax.set_ylabel('Score')
ax.set_title('Comparaison des modèles — Classification Catégorie')
ax.set_xticks(x)
ax.set_xticklabels(names, rotation=10)
ax.legend()
ax.set_ylim(0, 1.1)

for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{bar.get_height():.2%}',
            ha='center', va='bottom', fontsize=9)
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.02,
            f'{bar.get_height():.2%}',
            ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# MEILLEUR MODÈLE — CATÉGORIE
# ============================================================
best_name_cat = max(results_cat,
                    key=lambda x: results_cat[x]['f1_score'])
best_model_cat = results_cat[best_name_cat]['pipeline']
y_pred_cat = results_cat[best_name_cat]['y_pred']

print(f"🏆 Meilleur modèle : {best_name_cat}")
print(f"   Accuracy        : {results_cat[best_name_cat]['accuracy']:.4f}")
print(f"   F1-Score        : {results_cat[best_name_cat]['f1_score']:.4f}")
print(f"\n=== RAPPORT DÉTAILLÉ ===")
print(classification_report(
    y_test, y_pred_cat,
    target_names=le_cat.classes_
))

In [ ]:
# ============================================================
# MATRICE DE CONFUSION — CATÉGORIE
# ============================================================
cm_cat = confusion_matrix(y_test, y_pred_cat)

plt.figure(figsize=(14, 10))
sns.heatmap(cm_cat,
            annot=True, fmt='d', cmap='Blues',
            xticklabels=le_cat.classes_,
            yticklabels=le_cat.classes_)
plt.title(f'Matrice de Confusion — {best_name_cat}\n(Classification Catégorie)')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 5. Classification des Priorités

Même pipeline appliqué pour prédire la priorité :
- **P1_URGENT** — Critique, bloquant
- **P2_HIGH** — Problème important
- **P3_NORMAL** — Standard
- **P4_LOW** — Demande, information

In [ ]:
# ============================================================
# PRÉPARATION FEATURES ET LABELS — PRIORITÉ
# ============================================================
le_prio = LabelEncoder()
y_prio = le_prio.fit_transform(df['priorite'])

print(f"Classes priorité ({len(le_prio.classes_)}) :")
for i, cls in enumerate(le_prio.classes_):
    count = (df['priorite'] == cls).sum()
    print(f"  {i} — {cls:<15} : {count} tickets")

# Split 80/20
X_train_p, X_test_p, y_train_p, y_test_p = train_test_split(
    X, y_prio,
    test_size=0.2,
    random_state=42,
    stratify=y_prio
)

print(f"\n✅ Split effectué :")
print(f"   Train : {len(X_train_p)} tickets")
print(f"   Test  : {len(X_test_p)} tickets")

In [ ]:
# ============================================================
# ENTRAÎNEMENT 5 MODÈLES — PRIORITÉ
# ============================================================
results_prio = {}

print("=== ENTRAÎNEMENT — PRIORITÉ ===\n")

for name, model in models.items():
    with mlflow.start_run(run_name=f"priorite-{name}"):
        pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(**tfidf_params)),
            ('clf', clone(model))
        ])

        pipeline.fit(X_train_p, y_train_p)
        y_pred = pipeline.predict(X_test_p)

        acc = accuracy_score(y_test_p, y_pred)
        f1  = f1_score(y_test_p, y_pred, average='weighted')

        results_prio[name] = {
            'pipeline': pipeline,
            'accuracy': acc,
            'f1_score': f1,
            'y_pred': y_pred
        }

        # MLflow — on note ce run dans le carnet de bord
        mlflow.set_tag("cible", "priorite")
        mlflow.log_param("modele", name)
        mlflow.log_params(tfidf_params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)

        print(f"✅ {name:<25} Accuracy: {acc:.4f}  F1: {f1:.4f}")

# Meilleur modèle
best_name_prio  = max(results_prio,
                      key=lambda x: results_prio[x]['f1_score'])
best_model_prio = results_prio[best_name_prio]['pipeline']
y_pred_prio     = results_prio[best_name_prio]['y_pred']

print(f"\n🏆 Meilleur modèle : {best_name_prio}")
print(f"   Accuracy        : {results_prio[best_name_prio]['accuracy']:.4f}")
print(f"   F1-Score        : {results_prio[best_name_prio]['f1_score']:.4f}")
print(f"\n=== RAPPORT DÉTAILLÉ ===")
print(classification_report(
    y_test_p, y_pred_prio,
    target_names=le_prio.classes_
))

In [ ]:
# ============================================================
# MATRICE DE CONFUSION — PRIORITÉ
# ============================================================
cm_prio = confusion_matrix(y_test_p, y_pred_prio)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_prio,
            annot=True, fmt='d', cmap='Greens',
            xticklabels=le_prio.classes_,
            yticklabels=le_prio.classes_)
plt.title(f'Matrice de Confusion — {best_name_prio}\n(Classification Priorité)')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

---
## 6. Classification des Sous-Catégories

Même pipeline TF-IDF appliqué pour prédire la sous-catégorie.
Les labels ont été générés par LLM (Groq) dans le notebook 01.

- **39 classes** de sous-catégories
- Le modèle apprend à distinguer des classes plus fines
- Ex : dans "Matériel" → PC & Ordinateur / Imprimante / Caméra...

In [ ]:
# ============================================================
# PRÉPARATION FEATURES ET LABELS — SOUS-CATÉGORIE
# ============================================================

# Filtrer les sous-catégories avec < 2 tickets
# (train_test_split stratifié exige min. 2 membres par classe)
counts_sous  = df['sous_categorie'].value_counts()
sous_valides = counts_sous[counts_sous >= 2].index
df_sous      = df[df['sous_categorie'].isin(sous_valides)].copy()
X_sous       = df_sous['text_clean']

exclus = counts_sous[counts_sous < 2].index.tolist()
print(f"Sous-catégories exclues (< 2 tickets) : {exclus}")
print(f"Tickets conservés : {len(df_sous)} / {len(df)}")

le_sous = LabelEncoder()
y_sous  = le_sous.fit_transform(df_sous['sous_categorie'])

print(f"\nClasses sous-catégorie ({len(le_sous.classes_)}) :")
for i, cls in enumerate(le_sous.classes_):
    count = (df_sous['sous_categorie'] == cls).sum()
    print(f"  {i:2d} — {cls:<35} : {count} tickets")

X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
    X_sous, y_sous,
    test_size=0.2,
    random_state=42,
    stratify=y_sous
)

print(f"\n✅ Split effectué :")
print(f"   Train : {len(X_train_s)} tickets")
print(f"   Test  : {len(X_test_s)} tickets")

In [ ]:
# ============================================================
# ENTRAÎNEMENT 5 MODÈLES — SOUS-CATÉGORIE
# ============================================================
results_sous = {}

print("=== ENTRAÎNEMENT — SOUS-CATÉGORIE ===\n")

for name, model in models.items():
    with mlflow.start_run(run_name=f"sous_categorie-{name}"):
        pipeline = Pipeline([
            ('tfidf', TfidfVectorizer(**tfidf_params)),
            ('clf', clone(model))
        ])
        pipeline.fit(X_train_s, y_train_s)
        y_pred = pipeline.predict(X_test_s)

        acc = accuracy_score(y_test_s, y_pred)
        f1  = f1_score(y_test_s, y_pred, average='weighted')

        results_sous[name] = {
            'pipeline': pipeline,
            'accuracy': acc,
            'f1_score': f1,
            'y_pred': y_pred
        }

        # MLflow — on note ce run dans le carnet de bord
        mlflow.set_tag("cible", "sous_categorie")
        mlflow.log_param("modele", name)
        mlflow.log_params(tfidf_params)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)

        print(f"✅ {name:<25} Accuracy: {acc:.4f}  F1: {f1:.4f}")

# Meilleur modèle
best_name_sous  = max(results_sous, key=lambda x: results_sous[x]['f1_score'])
best_model_sous = results_sous[best_name_sous]['pipeline']
y_pred_sous     = results_sous[best_name_sous]['y_pred']

print(f"\n🏆 Meilleur modèle : {best_name_sous}")
print(f"   Accuracy        : {results_sous[best_name_sous]['accuracy']:.4f}")
print(f"   F1-Score        : {results_sous[best_name_sous]['f1_score']:.4f}")
print(f"\n=== RAPPORT DÉTAILLÉ ===")
print(classification_report(y_test_s, y_pred_sous, target_names=le_sous.classes_))

In [ ]:
# ============================================================
# MATRICE DE CONFUSION — SOUS-CATÉGORIE
# ============================================================
cm_sous = confusion_matrix(y_test_s, y_pred_sous)

plt.figure(figsize=(18, 14))
sns.heatmap(cm_sous,
            annot=True, fmt='d', cmap='Purples',
            xticklabels=le_sous.classes_,
            yticklabels=le_sous.classes_)
plt.title(f'Matrice de Confusion — {best_name_sous}\n(Classification Sous-Catégorie)')
plt.ylabel('Réel')
plt.xlabel('Prédit')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

---
## 7. Sauvegarde des Modèles

Les modèles sont sauvegardés au format `.pkl` pour être
chargés par le **AI Service FastAPI**.

### Fichiers sauvegardés
| Fichier | Description |
|---|---|
| model_categorie.pkl | Pipeline TF-IDF + classificateur catégorie (10 classes) |
| model_sous_categorie.pkl | Pipeline TF-IDF + classificateur sous-catégorie (39 classes) |
| model_priorite.pkl | Pipeline TF-IDF + classificateur priorité (4 classes) |
| label_encoder_cat.pkl | Encodeur labels catégorie |
| label_encoder_sous.pkl | Encodeur labels sous-catégorie |
| label_encoder_prio.pkl | Encodeur labels priorité |

In [ ]:
# ============================================================
# SAUVEGARDE MODÈLES
# ============================================================
save_dir = '../app/ai_service/models'
os.makedirs(save_dir, exist_ok=True)

models_to_save = {
    'model_categorie.pkl':      best_model_cat,
    'model_sous_categorie.pkl': best_model_sous,
    'model_priorite.pkl':       best_model_prio,
    'label_encoder_cat.pkl':    le_cat,
    'label_encoder_sous.pkl':   le_sous,
    'label_encoder_prio.pkl':   le_prio,
}

for filename, obj in models_to_save.items():
    with open(f'{save_dir}/{filename}', 'wb') as f:
        pickle.dump(obj, f)

print("✅ Modèles sauvegardés dans : backend/app/ai_service/models/")
for fname in sorted(os.listdir(save_dir)):
    if fname.endswith('.pkl'):
        size = os.path.getsize(f'{save_dir}/{fname}')
        print(f"   {fname:<40} {size/1024:.1f} KB")

---
## 7bis. Registre de modèles (MLflow Model Registry)

On enregistre les 3 modèles gagnants (un par cible) sous un **nom officiel et versionné**.
Contrairement au fichier `.pkl` (qui est écrasé à chaque nouvel entraînement), chaque
enregistrement ici crée une **nouvelle version** (v1, v2, v3...) sans perdre les précédentes —
on peut donc toujours revenir à une version antérieure si besoin.

In [ ]:
# ============================================================
# REGISTRE DE MODÈLES — versionner les 3 modèles gagnants
# ============================================================
import mlflow.sklearn

modeles_gagnants = [
    ("helpdesk-categorie",      best_model_cat,  results_cat[best_name_cat]['accuracy'],   results_cat[best_name_cat]['f1_score'],   best_name_cat),
    ("helpdesk-priorite",       best_model_prio, results_prio[best_name_prio]['accuracy'], results_prio[best_name_prio]['f1_score'], best_name_prio),
    ("helpdesk-sous_categorie", best_model_sous, results_sous[best_name_sous]['accuracy'], results_sous[best_name_sous]['f1_score'], best_name_sous),
]

for nom_registre, modele, acc, f1, algo in modeles_gagnants:
    with mlflow.start_run(run_name=f"registry-{nom_registre}"):
        mlflow.set_tag("algorithme", algo)
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.sklearn.log_model(
            sk_model=modele,
            name="model",
            registered_model_name=nom_registre,
            serialization_format="pickle",  # même méthode que le reste du projet (au lieu de 'skops', qui bloque XGBoost)
        )
        print(f"✅ {nom_registre:<28} enregistrée — algo={algo}, accuracy={acc:.4f}")

print("\n➡️  Ouvrez http://127.0.0.1:5000, menu 'Model registry' à gauche, pour voir les 3 modèles versionnés.")

---
## 7. Test de Prédiction

Validation du bon fonctionnement des modèles
sur des tickets fictifs représentatifs.

In [ ]:
# ============================================================
# FONCTION PRÉDICTION COMPLÈTE
# ============================================================
def predict_ticket(objet, description=""):
    text = clean_text(objet + " " + description)

    # Catégorie
    cat_enc   = best_model_cat.predict([text])[0]
    categorie = le_cat.inverse_transform([cat_enc])[0]

    # Sous-catégorie
    sous_enc      = best_model_sous.predict([text])[0]
    sous_categorie = le_sous.inverse_transform([sous_enc])[0]

    # Priorité
    prio_enc = best_model_prio.predict([text])[0]
    priorite = le_prio.inverse_transform([prio_enc])[0]

    # Confiance (LinearSVC decision function)
    try:
        scores     = best_model_cat.decision_function([text])[0]
        exp_scores = np.exp(scores - scores.max())
        confidence = f"{(exp_scores.max() / exp_scores.sum() * 100):.1f}%"
    except Exception:
        confidence = "N/A"

    return {
        'categorie':      categorie,
        'sous_categorie': sous_categorie,
        'priorite':       priorite,
        'confidence':     confidence,
    }

# Tests
test_tickets = [
    ("Problème réseau wifi bureau",    "Internet ne fonctionne pas"),
    ("Mot de passe outlook oublié",    "Je ne peux plus accéder à ma boite"),
    ("Panne ordinateur urgent",        "PC ne démarre plus, bloqué"),
    ("Installation sage paie",         "Nouveau employé à créer dans le système"),
    ("Problème imprimante",            "HP ne répond plus au réseau"),
    ("Demande accès VPN",              "Besoin VPN pour télétravail"),
    ("Réclamation client mall sfax",   "Client insatisfait du service"),
    ("Création compte messagerie",     "Nouveau employé besoin email"),
    ("Caméra extérieur en panne",      "NVR ne détecte plus la caméra"),
]

print(f"{'OBJET':<35} {'CATÉGORIE':<18} {'SOUS-CATÉGORIE':<30} {'PRIORITÉ':<12} {'CONF.'}")
print("-" * 110)
for objet, desc in test_tickets:
    r = predict_ticket(objet, desc)
    print(f"{objet:<35} {r['categorie']:<18} {r['sous_categorie']:<30} {r['priorite']:<12} {r['confidence']}")

---
## 8. Résumé Final

In [ ]:
# ============================================================
# RÉSUMÉ FINAL
# ============================================================
print("=" * 65)
print("  RÉSUMÉ — MODÈLES NLP HELPDESK IA GEISER")
print("=" * 65)

for label, results, le in [
    ("CATÉGORIE",      results_cat,  le_cat),
    ("SOUS-CATÉGORIE", results_sous, le_sous),
    ("PRIORITÉ",       results_prio, le_prio),
]:
    best = max(results, key=lambda x: results[x]['f1_score'])
    print(f"\n📊 CLASSIFICATION {label}")
    print(f"   Meilleur modèle : {best}")
    print(f"   Accuracy        : {results[best]['accuracy']*100:.2f}%")
    print(f"   F1-Score        : {results[best]['f1_score']:.4f}")
    print(f"   Nb classes      : {len(le.classes_)}")

print(f"\n✅ 6 fichiers sauvegardés dans : app/ai_service/models/")
print(f"\n🚀 Prochaine étape :")
print(f"   Notebook 03 — Routage automatique vers l'agent")
print("=" * 65)